<h1>Table of Contents<span class="tocSkip"></span></h1>
<div class="toc"><ul class="toc-item"></ul></div>

To call after the the API call and the creation of df

In [ ]:
## Algeria + Libya: Mazara del Vallo and Gela
it_identifiers = ['it-tso-0001itp-00074entry','it-tso-0001itp-00093entry']

dates_it = []
# hours_it = []
values_it = []
direction_it = []
operator_it = []
point_it = []
labels_it = []

urls = []
ids = []

for idt in it_identifiers:
    print(idt)
    try:
        url = 'https://transparency.entsog.eu/api/v1/operationalData?forceDownload=true&pointDirection={}&from=2023-03-06&to={}&indicator=Physical%20Flow&periodType=hour&timezone=CET&limit=-1&dataset=1&directDownload=true'.format(idt,today)
        r = requests.get(url)
        data = r.json()
        op = data['operationalData']

        for item in op:
            date = item['periodFrom'][:13]
#             hour = item['periodFrom'][11:13]
            value = item['value']
            drc = item['directionKey']
            opr = item['operatorKey']
            pt = item['pointKey']
            label = item['pointLabel']

            dates_it.append(date)
            hours_it.append(hour)
            values_it.append(value)
            direction_it.append(drc)
            operator_it.append(opr)
            point_it.append(pt)
            labels_it.append(label)
            urls.append(url)
            ids.append(idt)
            
    except Exception as e:
        print(e)
        print(idt)  

In [ ]:
df_it = pd.DataFrame()
df_it['dates'] = dates_it
# df_it['hours'] = hours_it
df_it['values'] = values_it
df_it['direction'] = direction_it
df_it['operator'] = operator_it
df_it['location'] = point_it
df_it['label'] = labels_it
df_it['country'] = df_it['operator'].str[:2]

In [ ]:
df_it.label.unique()

In [ ]:
df_it.set_index(pd.DatetimeIndex(df_it.dates), inplace=True)

In [ ]:
del df_it['dates']

In [ ]:
df_gela = df_it[df_it['label']=='Gela']
df_maz = df_it[df_it['label']=='Mazara del Vallo']

In [ ]:
idh = pd.date_range('2023-03-06 06:00:00', '2023-03-15 05:00:00',freq='H')

In [ ]:
df_gel = df_gela.reindex(idh, method='ffill')
df_ma = df_maz.reindex(idh, method='ffill')

In [ ]:
#we use the agg function with a dictionary of aggregation methods to resample the hourly data to daily data while preserving the first value of every string_col column.
df_gel_d = df_gel.resample('D').agg({'values': 'sum', 'direction': 'first','operator':'first','location':'first','label':'first','country':'first'})
df_ma_d = df_ma.resample('D').agg({'values': 'sum', 'direction': 'first','operator':'first','location':'first','label':'first','country':'first'})

In [ ]:
dff_it = pd.concat([df_gel_d,df_ma_d]).reset_index()

In [ ]:
dff_it.rename({'index':'dates'},axis=1,inplace=True)

In [ ]:
dff_it['dates'] = dff_it['dates'].dt.strftime('%Y-%m-%d')

In [ ]:
df = pd.concat([df,dff_it])